[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C23_Frontier_Alignment_Course/02_reward_modeling/02_reward_modeling.ipynb)

# 02 · 奖励建模与过优化（用 numpy 模拟）

目标：从合成偏好**从零训一个奖励模型**，再亲手造出 **over-optimization 的倒 U 曲线（Goodhart）**，并用 **RM 集成的分歧** 做保守奖励来缓解它。

路线：BT 损失与梯度 → 训 RM → proxy vs gold 倒 U → 长度偏置 hacking → RM 集成分歧 → 保守奖励 → ✏️ 练习 → 📖 答案 → 🧪 标度律胶囊。

> 心智模型：**回答 = 特征向量；gold 奖励 = 只看「真实质量」维；proxy RM = 从有限偏好学的线性打分器（带误差）。** 我们知道全部真相, 所以能精确地看 proxy 何时开始偏离 gold（钻空子）。

## 1 · Bradley-Terry 损失与梯度

RM 训练就是在「回答特征」上的成对 logistic 回归。偏好 $(y_w, y_l)$ 的损失：

$$\mathcal{L} = -\log\sigma(r_w - r_l), \quad r=w\cdot\phi$$

先实现损失与解析梯度, 并用**数值梯度对拍**验证求导正确（这是全模块训练的地基）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def sigmoid(z): return 1.0/(1.0+np.exp(-np.clip(z, -30, 30)))

def bt_loss_and_grad(w, phi_w, phi_l):
    '''单对偏好的 BT 负对数似然与对 w 的梯度。phi_*: 特征向量。'''
    delta = w @ (phi_w - phi_l)            # 分差 r_w - r_l
    loss = -np.log(sigmoid(delta))
    grad = -(1.0 - sigmoid(delta)) * (phi_w - phi_l)   # ∂L/∂w
    return loss, grad

# 数值梯度对拍
d = 4
w = rng.standard_normal(d)
phi_w = rng.standard_normal(d); phi_l = rng.standard_normal(d)
loss, grad = bt_loss_and_grad(w, phi_w, phi_l)
eps = 1e-6
num_grad = np.zeros(d)
for k in range(d):
    wp = w.copy(); wp[k] += eps
    wm = w.copy(); wm[k] -= eps
    num_grad[k] = (bt_loss_and_grad(wp, phi_w, phi_l)[0] - bt_loss_and_grad(wm, phi_w, phi_l)[0]) / (2*eps)
print('解析梯度:', np.round(grad, 4))
print('数值梯度:', np.round(num_grad, 4))
assert np.allclose(grad, num_grad, atol=1e-5), 'BT 梯度求导错误'
print('✅ BT 损失/梯度正确（解析 == 数值）')

## 2 · 从合成偏好训练奖励模型

设一个 **gold 质量函数**（=真实人类偏好, 只看前几维特征）, 按 BT 概率合成偏好对, 再用梯度下降训 RM。
验证训出的 RM 在 held-out 偏好上的预测准确率显著高于随机。

In [ ]:
D = 5
W_GOLD = np.array([2.0, 1.0, -1.0, 0.0, 0.0])   # gold 只在乎前 3 维
def gold_reward(phi): return phi @ W_GOLD

def sample_answer(rng): return rng.standard_normal(D)

def make_pref(rng):
    a, b = sample_answer(rng), sample_answer(rng)
    p_a = sigmoid(gold_reward(a) - gold_reward(b))
    return (a, b) if rng.random() < p_a else (b, a)   # (chosen, rejected)

def train_rm(prefs, D, lr=0.1, epochs=200):
    w = np.zeros(D)
    for _ in range(epochs):
        g = np.zeros(D)
        for pw, pl in prefs:
            _, gi = bt_loss_and_grad(w, pw, pl)
            g += gi
        w -= lr * g / len(prefs)
    return w

train_prefs = [make_pref(rng) for _ in range(500)]
test_prefs  = [make_pref(rng) for _ in range(500)]
w_rm = train_rm(train_prefs, D)

def pref_acc(w, prefs):
    return np.mean([w @ pw > w @ pl for pw, pl in prefs])
acc = pref_acc(w_rm, test_prefs)
print('学到的 RM 权重:', np.round(w_rm, 2))
print('gold 权重     :', W_GOLD)
print(f'held-out 偏好预测准确率 = {acc:.3f}')
assert acc > 0.7, 'RM 应学到与 gold 相关的偏好'
# RM 权重方向应与 gold 正相关
cos = (w_rm @ W_GOLD) / (np.linalg.norm(w_rm)*np.linalg.norm(W_GOLD))
assert cos > 0.8, 'RM 方向应接近 gold'
print(f'✅ RM 方向与 gold 余弦相似度 = {cos:.3f}, 学对了')

## 3 · 过优化的倒 U 曲线：proxy 升而 gold 降

**核心实验（复现 Gao et al. 2023）**：用一个**带误差的 proxy** 去「优化回答」, 同时用 **gold** 评估。

关键设置：**gold 是凹的**（真实质量有个最优点 `target`, 过犹不及）, 而 **proxy 误以为「沿某方向越走越好」**（它在质量维上大致对, 但还误奖励一个 gold 不在乎的「空子维」, 且会越过 gold 的最优点继续推）。沿 proxy 爬坡, 看 gold 先升后降。

In [ ]:
# gold: 凹函数, 在 target 处最优(过犹不及); 只看前3个「质量维」, 忽略「空子维」3,4
target = np.array([1.5, 1.0, -0.8, 0.0, 0.0])
def gold_reward_concave(phi):
    return -np.sum((phi[:3] - target[:3]) ** 2)      # 距最优点越远越差

# proxy: 质量维大致对, 但(a)会越过最优点继续推, (b)误奖励「空子维」3
w_proxy = np.array([1.0, 0.8, -0.6, 1.2, 0.0])
def proxy_reward(phi): return phi @ w_proxy          # 线性 -> 沿 w_proxy 永远「更高」

phi0 = np.zeros(D); phi = phi0.copy(); step = 0.15
kl_proxy, kl_gold, kls = [], [], []
for t in range(40):
    kls.append(np.linalg.norm(phi - phi0))           # 用位移当 KL 代理(离 ref 多远)
    kl_proxy.append(proxy_reward(phi))
    kl_gold.append(gold_reward_concave(phi))
    phi = phi + step * w_proxy                        # 沿 proxy 梯度上升(策略追 proxy 高分)

kl_proxy, kl_gold = np.array(kl_proxy), np.array(kl_gold)
peak = int(np.argmax(kl_gold))
print(f"{'步':>3}{'位移(KL)':>10}{'proxy':>9}{'gold':>9}")
for t in [0, 5, peak, 20, 30, 39]:
    print(f'{t:>3}{kls[t]:>10.2f}{kl_proxy[t]:>9.2f}{kl_gold[t]:>9.2f}')
print(f'\ngold 峰值在第 {peak} 步; proxy 峰值在第 {int(np.argmax(kl_proxy))} 步')
assert np.all(np.diff(kl_proxy) > 0), 'proxy 应单调上升(我们在直接优化它)'
assert 0 < peak < len(kl_gold) - 1, 'gold 应在中途见顶(先升后降)'
assert kl_gold[peak] > kl_gold[0], 'gold 早期确实上升(真改善)'
assert kl_gold[-1] < kl_gold[peak], '继续优化 proxy, gold 回落 = 过优化/Goodhart'
print('✅ 倒 U 出现：proxy 一路升, gold 见顶后回落。最优停点 = gold 峰, 远早于 proxy 峰')

**关键结论**：现实中你**看不到 gold**（它是真实人类偏好）, 只能看到一路上涨的 proxy。
所以「何时停」要靠**其他信号**辅助 —— KL（别走太远, 第4节）与**集成分歧**（第5节）。

## 4 · 一个具体的 hack：长度偏置

最经典的 reward hack：RM 把「长」误当「好」。我们让 proxy RM 对「长度特征」有正权重（gold 不在乎长度）, 沿 proxy 优化时回答会越来越长、proxy 虚高、gold 停滞。这把抽象的 Goodhart 落成可量的「长度漂移」。

In [ ]:
# 特征: [质量, 长度]。gold: 质量有最优点(q*=2, 凹), 完全不看长度。
#       proxy: 质量维也对, 但额外误奖励长度 -> 质量到顶后, 它靠「灌长度」继续刷分。
q_star = 2.0
def gold2(phi):  return -(phi[0] - q_star) ** 2                 # 只看质量, 过犹不及
def proxy2(phi): return -(phi[0] - q_star) ** 2 + 0.8 * phi[1]  # 质量 + 误奖励长度

phi = np.array([0.0, 0.0])   # [质量, 长度]
quality_hist, lengths, gold_hist, proxy_hist = [], [], [], []
for t in range(25):
    quality_hist.append(phi[0]); lengths.append(phi[1])
    gold_hist.append(gold2(phi)); proxy_hist.append(proxy2(phi))
    grad = np.array([-2 * (phi[0] - q_star), 0.8])   # 沿 proxy 梯度: 质量趋向 q*, 长度无限涨
    phi = phi + 0.15 * grad
quality_hist, lengths, gold_hist, proxy_hist = map(np.array, (quality_hist, lengths, gold_hist, proxy_hist))

print(f"{'步':>3}{'质量':>8}{'长度':>8}{'proxy':>8}{'gold':>8}")
for t in (0, 6, 12, 18, 24):
    print(f'{t:>3}{quality_hist[t]:>8.2f}{lengths[t]:>8.2f}{proxy_hist[t]:>8.2f}{gold_hist[t]:>8.2f}')

# 诊断: 后期(质量已饱和)proxy 仍升、gold 却平 -> 涨分全来自灌长度 = 长度 hack 指纹
late_proxy_gain = proxy_hist[-1] - proxy_hist[-8]
late_gold_gain  = gold_hist[-1]  - gold_hist[-8]
late_len_gain   = lengths[-1]    - lengths[-8]
print(f'\n后8步: proxy 涨 {late_proxy_gain:.2f}, gold 涨 {late_gold_gain:.4f}(≈0), 长度涨 {late_len_gain:.2f}')
assert lengths[-1] > lengths[0] + 2, '回答应越来越长'
assert late_proxy_gain > 0.5, '后期 proxy 仍在涨'
assert abs(late_gold_gain) < 0.05, '后期 gold 已饱和(质量到顶, 长度不算分)'
assert abs(late_proxy_gain - 0.8 * late_len_gain) < 0.05, 'proxy 后期涨分几乎全来自长度×0.8'
print('✅ 长度 hack 现形：质量饱和后, proxy 靠灌长度虚高, gold 纹丝不动')

## 5 · RM 集成：分布外分歧 > 分布内分歧

训多个 RM（不同数据子集）。在**训练分布内**它们应打分一致（分歧小）；在**分布外**（远离训练数据的回答）它们各自外推、分歧大。这个分歧就是「RM 没把握」的信号。

In [ ]:
def train_ensemble(M, n_each, D, rng):
    rms = []
    for _ in range(M):
        prefs = [make_pref(rng) for _ in range(n_each)]
        rms.append(train_rm(prefs, D, epochs=200))
    return rms

ensemble = train_ensemble(M=5, n_each=60, D=D, rng=rng)

def ens_mean_std(rms, phi):
    scores = np.array([r @ phi for r in rms])
    return scores.mean(), scores.std()

# 分布内: 标准正态采样(和训练同分布); 分布外: 放大到远处
in_dist  = [sample_answer(rng) for _ in range(200)]
out_dist = [sample_answer(rng) * 6.0 for _ in range(200)]   # 远离训练分布
std_in  = np.mean([ens_mean_std(ensemble, phi)[1] for phi in in_dist])
std_out = np.mean([ens_mean_std(ensemble, phi)[1] for phi in out_dist])
print(f'分布内平均分歧(std) = {std_in:.3f}')
print(f'分布外平均分歧(std) = {std_out:.3f}')
assert std_out > std_in * 1.5, '分布外分歧应明显更大'
print('✅ 集成分歧是「RM 在外推」的信号：分布外 RM 们各执一词')

## 6 · 保守奖励：均值 − λ·分歧

用集成造保守奖励 $r_{cons}=\bar r - \lambda s$：分歧大处主动扣分, 逼优化器别去钻没把握的角落。
验证：沿**保守奖励**优化, 比沿**单个 proxy**优化, 在远处(分歧大)更早被「刹住」, 从而少踩过优化的坑。

In [ ]:
def conservative_reward(rms, phi, lam):
    m, s = ens_mean_std(rms, phi)
    return m - lam * s

# 对比两条优化轨迹的终点位移(越大越冒进 -> 越易过优化)
def optimize(reward_fn, steps=30, step=0.15):
    phi = np.zeros(D)
    for _ in range(steps):
        g = np.zeros(D); eps = 1e-3
        for k in range(D):
            pp = phi.copy(); pp[k]+=eps; pm = phi.copy(); pm[k]-=eps
            g[k] = (reward_fn(pp)-reward_fn(pm))/(2*eps)
        phi = phi + step * g
    return phi

single = ensemble[0]
phi_greedy = optimize(lambda p: single @ p)                       # 追单个 RM
phi_cons   = optimize(lambda p: conservative_reward(ensemble, p, lam=2.0))  # 保守
disp_greedy = np.linalg.norm(phi_greedy)
disp_cons   = np.linalg.norm(phi_cons)
print(f'贪婪(追单RM)终点位移 = {disp_greedy:.2f}')
print(f'保守(均值-λ分歧)位移 = {disp_cons:.2f}')
assert disp_cons < disp_greedy, '保守奖励应抑制走向高分歧远处'
print('✅ 保守奖励把策略「刹」在 RM 仍有把握的范围内, 延缓过优化')

---
## ✏️ 练习 1：批量 BT 损失 `batch_bt_loss`

实现 `batch_bt_loss(w, prefs)`：返回一批偏好对的**平均** BT 负对数似然（标量）。用于监控 RM 训练。

In [ ]:
def batch_bt_loss(w, prefs):
    # TODO: 对每对 (pw,pl) 算 -log sigmoid(w@(pw-pl)), 返回平均
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
L = batch_bt_loss(w_rm, test_prefs)
L0 = batch_bt_loss(np.zeros(D), test_prefs)
assert L > 0, '损失应为正'
assert abs(L0 - np.log(2)) < 0.02, 'w=0 时每对损失 ≈ log2'
assert L < L0, '训过的 RM 损失应低于零初始化'
print(f'训过的 RM 平均 BT 损失 = {L:.3f} < 零初始化 {L0:.3f} (=log2)')
print('✅ 练习 1 通过')

## ✏️ 练习 2：从零训 RM `train_rm_sgd`

实现一个 **mini-batch SGD** 版的 RM 训练 `train_rm_sgd(prefs, D, lr, epochs, batch, rng)`：每 epoch 打乱、按 batch 累加梯度更新。返回 w。验证准确率达标。

In [ ]:
def train_rm_sgd(prefs, D, lr=0.2, epochs=50, batch=32, rng=None):
    # TODO: w=0; 每 epoch 打乱 prefs, 按 batch 取子集, 累加 bt 梯度并更新
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
w2 = train_rm_sgd(train_prefs, D, rng=np.random.default_rng(1))
acc2 = pref_acc(w2, test_prefs)
assert acc2 > 0.7, 'SGD 训的 RM 也应达标'
cos2 = (w2 @ W_GOLD)/(np.linalg.norm(w2)*np.linalg.norm(W_GOLD))
assert cos2 > 0.8, '方向应接近 gold'
print(f'SGD-RM 准确率={acc2:.3f}, 与 gold 余弦={cos2:.3f}')
print('✅ 练习 2 通过：mini-batch SGD 也能训出对的 RM')

## ✏️ 练习 3：reward hacking 检测 `detect_hacking`

给一条优化轨迹的 (proxy 序列, gold 序列), 检测是否发生过优化。
实现 `detect_hacking(proxy_hist, gold_hist)`：若存在某点之后 **proxy 仍升但 gold 已降**, 返回该转折点下标; 否则返回 -1。

In [ ]:
def detect_hacking(proxy_hist, gold_hist):
    # TODO: 找 gold 的峰值下标 peak; 若 peak 后 proxy 末值 > proxy[peak]
    #       且 gold 末值 < gold[peak], 返回 peak; 否则 -1
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pk = detect_hacking(list(kl_proxy), list(kl_gold))
assert pk != -1 and pk == int(np.argmax(kl_gold)), '应检测到过优化转折点'
# 一条健康轨迹(proxy 与 gold 同升)应返回 -1
healthy_p = [1.0, 2.0, 3.0, 4.0]; healthy_g = [1.0, 1.8, 2.5, 3.1]
assert detect_hacking(healthy_p, healthy_g) == -1, '同升轨迹不算 hacking'
print(f'检测到过优化转折点 @ 第 {pk} 步; 健康轨迹返回 -1')
print('✅ 练习 3 通过：能从 proxy/gold 轨迹诊断过优化')

## ✏️ 练习 4：集成分歧 `disagreement`

实现 `disagreement(rms, phi)`：返回集成在 phi 上打分的标准差（认知不确定性）。
再验证：把它当奖励惩罚项, 高分歧回答的保守奖励确实被压低。

In [ ]:
def disagreement(rms, phi):
    # TODO: 返回 [r@phi for r in rms] 的标准差
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
phi_in  = sample_answer(np.random.default_rng(2))
phi_out = phi_in * 6.0
s_in, s_out = disagreement(ensemble, phi_in), disagreement(ensemble, phi_out)
assert s_out > s_in, '远处分歧应更大'
# 保守奖励在高分歧处被压得更狠
lam = 2.0
penalty_in  = lam * s_in
penalty_out = lam * s_out
assert penalty_out > penalty_in, '高分歧 -> 更大惩罚'
print(f'分布内分歧={s_in:.3f}(罚 {penalty_in:.2f}), 分布外={s_out:.3f}(罚 {penalty_out:.2f})')
print('✅ 练习 4 通过：分歧度量不确定性, 驱动保守奖励')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def batch_bt_loss(w, prefs):
    losses = [-np.log(sigmoid(w @ (pw - pl))) for pw, pl in prefs]
    return float(np.mean(losses))

In [ ]:
# 练习 2 参考答案
def train_rm_sgd(prefs, D, lr=0.2, epochs=50, batch=32, rng=None):
    rng = rng or np.random.default_rng(0)
    w = np.zeros(D)
    idx = np.arange(len(prefs))
    for _ in range(epochs):
        rng.shuffle(idx)
        for s in range(0, len(prefs), batch):
            g = np.zeros(D)
            chunk = idx[s:s+batch]
            for j in chunk:
                pw, pl = prefs[j]
                _, gi = bt_loss_and_grad(w, pw, pl)
                g += gi
            w -= lr * g / len(chunk)
    return w

In [ ]:
# 练习 3 参考答案
def detect_hacking(proxy_hist, gold_hist):
    peak = int(np.argmax(gold_hist))
    if peak < len(gold_hist) - 1 and proxy_hist[-1] > proxy_hist[peak] and gold_hist[-1] < gold_hist[peak]:
        return peak
    return -1

In [ ]:
# 练习 4 参考答案
def disagreement(rms, phi):
    return float(np.std([r @ phi for r in rms]))

---
## 🧪 真实数据胶囊：过优化的标度律（Gao et al. 2023）

Gao et al. 2023 发现 gold 分随 KL 的变化可用**简洁函数族**刻画。论文的 best-of-n 形式近似为：

$$R_{gold}(d) \approx d\,(\alpha - \beta \ln d), \quad d=\sqrt{\mathrm{KL}}$$

（αβ 为拟合系数）。这是一条先升后降的曲线。我们用论文报告量级的系数复现它, 并求解析最优 KL。

In [ ]:
# 论文 best-of-n 形式: R_gold(d) = d*(alpha - beta*ln d), d=sqrt(KL)
# 用接近论文量级的玩具系数(真实系数随 RM 大小变化)
alpha, beta = 2.0, 0.5
def gold_vs_d(d): return d * (alpha - beta * np.log(d))

# 解析最优: dR/dd = alpha - beta*ln d - beta = 0 -> ln d = (alpha-beta)/beta
d_opt = np.exp((alpha - beta) / beta)
ds = np.linspace(0.2, 2.5 * d_opt, 200)   # 取够宽, 覆盖解析最优
gold = gold_vs_d(ds)
d_peak = ds[np.argmax(gold)]
print(f'数值峰 d≈{d_peak:.2f}, KL≈{d_peak**2:.1f}')
print(f'解析最优 d*={d_opt:.2f}, KL*={d_opt**2:.1f}')
assert abs(d_peak - d_opt) < 0.5, '数值峰应与解析最优一致'
assert gold[-1] < gold.max(), '大 KL 处 gold 回落(过优化)'
print('✅ 复现标度律倒 U；存在解析最优 KL —— 优化到此为止最好')

**🧪 胶囊练习**：实现 `optimal_kl(alpha, beta)`：用解析解 $\ln d^* = (\alpha-\beta)/\beta$ 返回**最优 KL**（=$d^{*2}$）。
并验证更大的 RM（论文中对应更大 α、更小 β）允许更大的最优 KL —— 即更晚过优化。

In [ ]:
def optimal_kl(alpha, beta):
    # TODO: d_opt = exp((alpha-beta)/beta); 返回 d_opt**2
    raise NotImplementedError

In [ ]:
# 自测
kl_small = optimal_kl(2.0, 0.5)      # 较小 RM
kl_big   = optimal_kl(3.0, 0.3)      # 较大 RM(更忠实): 更大 alpha, 更小 beta
assert abs(kl_small - np.exp((2.0-0.5)/0.5)**2) < 1e-6
assert kl_big > kl_small, '更忠实的 RM 允许优化更久才过优化'
print(f'小 RM 最优 KL ≈ {kl_small:.1f}; 大 RM 最优 KL ≈ {kl_big:.1f}')
print('✅ 胶囊练习通过：RM 越好, 可安全优化的 KL 越大')

In [ ]:
# 📖 胶囊参考答案
def optimal_kl(alpha, beta):
    d_opt = np.exp((alpha - beta) / beta)
    return d_opt ** 2

### 小结
- **RM = 成对 logistic 回归**：BT 损失 $-\log\sigma(r_w-r_l)$, 只约束分差。
- **过优化(Goodhart)**：对 proxy 优化过头, gold 先升后降(倒 U)。最优停点 = gold 峰, 远早于 proxy 峰。
- **KL 正则**：把策略拴在 RM 仍可信的邻域(信任域), 延缓但不根治过优化。
- **RM 集成**：分歧 = 认知不确定性; 保守奖励 $\bar r-\lambda s$ 在没把握处变保守。但抓不住共有盲区。

下一站：**模块 03 · 可扩展监督** —— 当人类(乃至 RM)都难以可靠评判时, 怎么靠 debate/分解放大监督。